# 多头注意力

用独立学习得到的$h$组不同的
*线性投影*（linear projections）来变换查询、键和值。
然后，这$h$组变换后的查询、键和值将并行地送到注意力汇聚中。
最后，将这$h$个注意力汇聚的输出拼接在一起，
并且通过另一个可以学习的线性投影进行变换，
以产生最终输出。

给定查询$\mathbf{q} \in \mathbb{R}^{d_q}$、
键$\mathbf{k} \in \mathbb{R}^{d_k}$和
值$\mathbf{v} \in \mathbb{R}^{d_v}$，
每个注意力头$\mathbf{h}_i$（$i = 1, \ldots, h$）的计算方法为：

$$\mathbf{h}_i = f(\mathbf W_i^{(q)}\mathbf q, \mathbf W_i^{(k)}\mathbf k,\mathbf W_i^{(v)}\mathbf v) \in \mathbb R^{p_v},$$

其中，可学习的参数包括
$\mathbf W_i^{(q)}\in\mathbb R^{p_q\times d_q}$、
$\mathbf W_i^{(k)}\in\mathbb R^{p_k\times d_k}$和
$\mathbf W_i^{(v)}\in\mathbb R^{p_v\times d_v}$，
以及代表注意力汇聚的函数$f$。

多头注意力的输出需要经过另一个线性转换，
它对应着$h$个头连结后的结果，因此其可学习参数是
$\mathbf W_o\in\mathbb R^{p_o\times h p_v}$：

$$\mathbf W_o \begin{bmatrix}\mathbf h_1\\\vdots\\\mathbf h_h\end{bmatrix} \in \mathbb{R}^{p_o}.$$

基于这种设计，每个头都可能会关注输入的不同部分，
可以表示比简单加权平均值更复杂的函数。

In [ ]:
import math
import torch
from torch import nn

将查询、键和值的线性变换的输出数量设置为   
$p_q h = p_k h = p_v h = p_o$，
则可以并行计算$h$个头。

In [ ]:
def transpose_qkv(X, num_heads):
    batch_size, seq_len, num_hiddens = X.shape
    head_dim = num_hiddens // num_heads

    X = X.reshape(batch_size, seq_len, num_heads, head_dim)
    # (B, seq_len, num_heads, head_dim)
    X = X.permute(0, 2, 1, 3)
    # (B, num_heads, seq_len, head_dim)
    X = X.reshape(-1, seq_len, head_dim)
    # (B * num_heads, seq_len, head_dim)

    return X



In [ ]:
def transpose_output(X, num_heads):
    # X.shape = (batch_size * num_heads, seq_len, head_dim)
    batch_size = X.shape[0] // num_heads
    seq_len = X.shape[1]
    head_dim = X.shape[2]

    X = X.reshape(batch_size, num_heads, seq_len, head_dim)
    # (B, num_heads, seq_len, head_dim)
    X = X.permute(0, 2, 1, 3)
    # (B, seq_len, num_heads, head_dim)
    X = X.reshape(batch_size, seq_len, -1)
    # (B, seq_len, num_heads * head_dim)

    return X

In [ ]:
from typing import Any


class MultiheadAttention(nn.Module):
    def __init__(self,key_size,value_size,query_size,num_hiddens,num_heads,dropout,bias = False, **kwargs: Any) -> None:
        super(MultiheadAttention,self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = DotProductAttetion(dropout)
        # 在上一个里有过了
        self.W_q = nn.Linear(query_size, num_hiddens, bias = bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias = bias)
        self.W_k = nn.Linear(key_size, num_hiddens,  bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):

        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)

        if valid_lens is not None:
            # 在轴0，将第一项复制num_heads次，
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)
            
        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)